# 📊 Báo Cáo Đánh Giá Mô Hình Phân Loại Rác
Notebook này chạy độc lập để load mô hình SVM và Ensemble (V2), đánh giá chi tiết trên tập Validation và xuất ra các biểu đồ phục vụ cho Báo cáo / Đồ án.

In [ ]:
import os
import sys
import cv2
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Do notebook nằm trong thư mục notebooks/, ta phải lui ra ngoài 1 thư mục (..) để lấy project_root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.core.config import CLASSES
from src.scripts.feature_utils import extract_features

print('✅ Đã load thư viện và cấu hình đường dẫn thành công!')

### 1. Hàm Load Data Validation (Chạy Đa Luồng)
Trích xuất 1878 features (HSV + HOG + LBP) từ tập Validation (không cần Augmentation khi Validation) để tiến hành đánh giá.

In [ ]:
import concurrent.futures
from tqdm.notebook import tqdm

IMG_SIZE = (64, 64)

def _process_single_image(file_path, class_idx):
    try:
        img = cv2.imread(file_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)
        # Sử dụng đúng thuật toán trích xuất của V2
        features = extract_features(img, use_grabcut=False)
        return features, class_idx
    except Exception:
        return None, None

def load_validation_data(data_dir):
    X, y = [], []
    print(f'📂 Đang tải tập Validation từ: {data_dir}')
    for class_idx, class_name in enumerate(CLASSES):
        class_path = os.path.join(data_dir, class_name)
        if not os.path.exists(class_path): continue
        
        files = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        file_paths = [os.path.join(class_path, f) for f in files]
        
        with concurrent.futures.ProcessPoolExecutor() as executor:
            futures = [executor.submit(_process_single_image, fp, class_idx) for fp in file_paths]
            for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc=f'{class_name:15s}'):
                feat, label = future.result()
                if feat is not None:
                    X.append(feat)
                    y.append(label)
                    
    return np.array(X, dtype=np.float32), np.array(y)

val_dir = os.path.join(project_root, 'dataset', 'validation')
X_val, y_val = load_validation_data(val_dir)
print(f'\n✅ Đã load xong: {len(X_val)} ảnh validation.')

### 2. Tải Mô Hình & Đánh Giá Chi Tiết
Đánh giá cả 2 cấu hình (SVM V2 và Ensemble V2) để so sánh hiệu năng.

In [ ]:
def evaluate_model(model_path, model_name):
    if not os.path.exists(model_path):
        print(f'❌ Không tìm thấy model tại: {model_path}')
        return
        
    print(f'\n{"="*50}\n🚀 ĐÁNH GIÁ MÔ HÌNH: {model_name}\n{"="*50}')
    bundle = joblib.load(model_path)
    model = bundle['model']
    scaler = bundle['scaler']
    
    # Chuẩn hóa Data
    X_val_scaled = scaler.transform(X_val)
    
    # Dự đoán
    y_pred = model.predict(X_val_scaled)
    
    # In Classification Report
    print('\n📊 BÁO CÁO PHÂN LOẠI (Classification Report):')
    print(classification_report(y_val, y_pred, target_names=CLASSES))
    
    # Vẽ Confusion Matrix
    cm = confusion_matrix(y_val, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('Thực tế (True Label)')
    plt.xlabel('Dự đoán (Predicted Label)')
    plt.show()

# Đánh giá SVM V2
svm_path = os.path.join(project_root, 'outputs', 'svm_results', 'svm_v2_model.pkl')
evaluate_model(svm_path, 'SVM V2 (HSV + HOG + LBP)')

# Đánh giá Ensemble V2
ensemble_path = os.path.join(project_root, 'outputs', 'ensemble_results', 'best_ensemble_model.pkl')
evaluate_model(ensemble_path, 'Soft Voting Ensemble V2 (SVM+RF+KNN)')

### 3. Trực quan hóa (Hiển thị ngẫu nhiên 5 ảnh bị sai)
Trực quan hóa xem mô hình Ensemble đang bị nhầm lẫn ở những bức ảnh nào, và tại sao.

In [ ]:
import random

def show_mistakes(model_path):
    if not os.path.exists(model_path): return
    bundle = joblib.load(model_path)
    model, scaler = bundle['model'], bundle['scaler']
    X_val_scaled = scaler.transform(X_val)
    y_pred = model.predict(X_val_scaled)
    
    # Tìm các index đoán sai
    wrong_indices = np.where(y_pred != y_val)[0]
    if len(wrong_indices) == 0:
        print('Tuyệt vời! Không có ảnh nào đoán sai.')
        return
        
    # Chọn ngẫu nhiên tối đa 5 ảnh sai
    sample_indices = random.sample(list(wrong_indices), min(5, len(wrong_indices)))
    
    # Tìm đường dẫn file gốc của các ảnh validation để hiển thị
    val_files = []
    for class_name in CLASSES:
        class_path = os.path.join(val_dir, class_name)
        if os.path.exists(class_path):
            files = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            val_files.extend(files)
            
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle('Một số trường hợp Mô hình dự đoán SAI', fontsize=16)
    
    for idx, ax in zip(sample_indices, axes):
        true_label = CLASSES[y_val[idx]]
        pred_label = CLASSES[y_pred[idx]]
        
        img = cv2.cvtColor(cv2.imread(val_files[idx]), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f'True: {true_label}\nPred: {pred_label}', color='red')
        ax.axis('off')
    plt.show()

show_mistakes(ensemble_path)